## Loading Files

In [1]:
from safetensors import safe_open
import os

files = ["model-00001-of-00002.safetensors", "model-00002-of-00002.safetensors"]

for file in files:
    print(f"\n=== {file} ===")

    with safe_open(f"./my_models/{file}", framework="pt", device="cpu") as f:
        for key in f.keys():
            print(f"Name: {key}, shape:{f.get_tensor(key).shape}")


=== model-00001-of-00002.safetensors ===
Name: model.embed_tokens.weight, shape:torch.Size([128256, 3072])
Name: model.layers.0.input_layernorm.weight, shape:torch.Size([3072])
Name: model.layers.0.mlp.down_proj.weight, shape:torch.Size([3072, 8192])
Name: model.layers.0.mlp.gate_proj.weight, shape:torch.Size([8192, 3072])
Name: model.layers.0.mlp.up_proj.weight, shape:torch.Size([8192, 3072])
Name: model.layers.0.post_attention_layernorm.weight, shape:torch.Size([3072])
Name: model.layers.0.self_attn.k_proj.weight, shape:torch.Size([1024, 3072])
Name: model.layers.0.self_attn.o_proj.weight, shape:torch.Size([3072, 3072])
Name: model.layers.0.self_attn.q_proj.weight, shape:torch.Size([3072, 3072])
Name: model.layers.0.self_attn.v_proj.weight, shape:torch.Size([1024, 3072])
Name: model.layers.1.input_layernorm.weight, shape:torch.Size([3072])
Name: model.layers.1.mlp.down_proj.weight, shape:torch.Size([3072, 8192])
Name: model.layers.1.mlp.gate_proj.weight, shape:torch.Size([8192, 3072

## First layer Weights

In [2]:
weights = {}

for file in files:
    with safe_open(f"./my_models/{file}", framework="pt", device="cpu") as f:
        for key in f.keys():
            if "layers.0." in key:
                weights[key] = f.get_tensor(key)

for name, tensor in weights.items():
    print(name, tensor.shape)

model.layers.0.input_layernorm.weight torch.Size([3072])
model.layers.0.mlp.down_proj.weight torch.Size([3072, 8192])
model.layers.0.mlp.gate_proj.weight torch.Size([8192, 3072])
model.layers.0.mlp.up_proj.weight torch.Size([8192, 3072])
model.layers.0.post_attention_layernorm.weight torch.Size([3072])
model.layers.0.self_attn.k_proj.weight torch.Size([1024, 3072])
model.layers.0.self_attn.o_proj.weight torch.Size([3072, 3072])
model.layers.0.self_attn.q_proj.weight torch.Size([3072, 3072])
model.layers.0.self_attn.v_proj.weight torch.Size([1024, 3072])


## Rebuilding Modules

In [58]:
import torch


class LlamaAttention(torch.nn.Module):
    def __init__(self, num_kv_head, num_attn_head, head_dim, hidden_size, max_seq_len):
        super().__init__()

        self.num_kv_head = num_kv_head
        self.num_attn_head = num_attn_head
        self.head_dim = head_dim
        self.hidden_size = hidden_size
        self.max_seq_len = max_seq_len

        self.theta = 500000
        self.low_freq_factor = 1.0
        self.high_freq_factor = 4.0
        self.factor = 32.0
        self.original_max_position_embeddings = 8192

        self.k_proj = torch.nn.Linear(
            in_features=hidden_size, out_features=num_kv_head * head_dim
        )
        self.o_proj = torch.nn.Linear(in_features=hidden_size, out_features=hidden_size)
        self.q_proj = torch.nn.Linear(in_features=hidden_size, out_features=hidden_size)
        self.v_proj = torch.nn.Linear(
            in_features=hidden_size, out_features=num_kv_head * head_dim
        )

        self.register_buffer(
            "mask", torch.triu(torch.ones(max_seq_len, max_seq_len), diagonal=1).bool()
        )

    def forward(self, x):

        B, S, D = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, S, self.num_attn_head, self.head_dim).transpose(1, 2)
        k = k.view(B, S, self.num_kv_head, self.head_dim).transpose(1, 2)
        v = v.view(B, S, self.num_kv_head, self.head_dim).transpose(1, 2)

        # Llama 3.1 RoPE
        # Base inverse frequencies
        inv_freq = 1.0 / (
            self.theta
            ** (
                torch.arange(
                    0,
                    self.head_dim,
                    2,
                    device=x.device,
                    dtype=torch.float32,
                )
                / self.head_dim
            )
        )

        # Wavelength of each frequency
        wavelen = 2 * torch.pi / inv_freq

        low_freq_wavelen = self.original_max_position_embeddings / self.low_freq_factor

        high_freq_wavelen = (
            self.original_max_position_embeddings / self.high_freq_factor
        )

        # Long-wavelength frequencies:
        # divide their inverse frequency by the scaling factor.
        inv_freq_llama = torch.where(
            wavelen > low_freq_wavelen,
            inv_freq / self.factor,
            inv_freq,
        )

        # Frequencies in the transition region
        smooth_factor = (
            self.original_max_position_embeddings / wavelen - self.low_freq_factor
        ) / (self.high_freq_factor - self.low_freq_factor)

        smoothed_inv_freq = (
            1 - smooth_factor
        ) * inv_freq_llama / self.factor + smooth_factor * inv_freq_llama

        # Transition region:
        # high_freq_wavelen <= wavelen <= low_freq_wavelen
        is_medium_freq = ~(wavelen < high_freq_wavelen) & ~(wavelen > low_freq_wavelen)

        inv_freq_llama = torch.where(
            is_medium_freq,
            smoothed_inv_freq,
            inv_freq_llama,
        )

        # Position × frequency
        positions = torch.arange(
            S,
            device=x.device,
            dtype=torch.float32,
        )

        freqs = torch.outer(
            positions,
            inv_freq_llama,
        )

        # cos/sin
        cos = freqs.cos()
        sin = freqs.sin()

        # [S, 64] → [S, 128]
        cos = torch.cat((cos, cos), dim=-1)
        sin = torch.cat((sin, sin), dim=-1)

        # Apply RoPE
        q = (q * cos) + (self.rotate_half(q) * sin)
        k = (k * cos) + (self.rotate_half(k) * sin)

        num_groups = self.num_attn_head // self.num_kv_head

        k = k.repeat_interleave(num_groups, dim=1)
        v = v.repeat_interleave(num_groups, dim=1)

        scores = torch.einsum("bhqd, bhkd -> bhqk", q, k) * (self.head_dim**-0.5)
        scores = scores.masked_fill(self.mask[:S, :S], float("-inf"))

        print("x:", x.shape)
        print("q:", q.shape)
        print("k:", k.shape)
        print("v:", v.shape)
        print("A:", scores.shape)

    def rotate_half(self, x):
        x1 = x[..., : self.head_dim // 2]
        x2 = x[..., self.head_dim // 2 :]

        return torch.cat((-x2, x1), dim=-1)

In [57]:
hidden_size = 3072
head_dim = 128
num_attn_head = 24
num_kv_head = 8
max_seq_len = 100
theta = 500000

gqa_atten = LlamaAttention(
    num_kv_head, num_attn_head, head_dim, hidden_size, max_seq_len
)

x = torch.randn((4, 10, hidden_size))
gqa_atten(x)

x: torch.Size([4, 10, 3072])
q: torch.Size([4, 24, 10, 128])
k: torch.Size([4, 24, 10, 128])
v: torch.Size([4, 24, 10, 128])
A: torch.Size([4, 24, 10, 10])
